In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path




In [3]:
try:
    eng = engs.get_engine()
    text_qry = text(engs.load_query('qry_olos.sql'))
    df = pd.read_sql(text_qry, eng)
except Exception as e:
    print (f'Erro ao executar consulta: {e}')

In [4]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [5]:
df_today = df.copy()

df_today = df_today.groupby(['area','base_type']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
df_today['tx'] = ((df_today['atendidas'] /  df_today['tentativas']) * 100 ).round(2)
df_totay = df_today[df_today['tentativas'] > 100]
df_today.sort_values(by='tx',ascending=False)


,area,base_type,tentativas,atendidas,tx
12,Nutrição,evento,1305,59,4.52
22,Veterinária,inativa,854,38,4.45
2,Enfermagem,evento,2365,86,3.64
14,Outras Áreas,Carrinho,2970,105,3.54
4,Fisioterapia,Base Leads,15370,511,3.32
6,Fisioterapia,Material Rico,983,31,3.15
11,Multi,evento,830,25,3.01
18,Psicologia,Base Leads,8185,238,2.91
10,Medicina,evento,705,20,2.84
0,Enfermagem,Material Rico,1745,49,2.81


In [6]:
df_tab = df.copy()
df_tab = df_tab[df_tab['data'] == '2026-05-13']
df_tab = df_tab.groupby(['disposition_nivel_1']).agg(
    quantidade = ('atendidas','sum')
).reset_index()
df_tab['total'] = df_tab['quantidade'].sum()
df_tab['representativada'] = ((df_tab['quantidade'] / df_tab['total']) * 100).round(2)
df_tab.sort_values(by='quantidade',ascending=False)


,disposition_nivel_1,quantidade,total,representativada
12,IMPRODUTIVO LIGACAO CAIU,53,161,32.92
23,RETORNO AGENDADO,37,161,22.98
8,IMPRODUTIVO CAIXA POSTAL NAO ATENDE,12,161,7.45
14,IMPRODUTIVO LIGACAO SEM AUDIO,11,161,6.83
24,RETORNO OUTRA PLATAFORMA,10,161,6.21
10,IMPRODUTIVO DESLIGOU NA APRESENTACAO,9,161,5.59
19,RECUSA NAO INFORMOU,5,161,3.11
16,NEGOCIACAO EM ANDAMENTO,4,161,2.48
22,RECUSA SEM TEMPO PARA SE DEDICAR,3,161,1.86
18,RECUSA FINANCEIRO,3,161,1.86
